<a href="https://colab.research.google.com/gist/kmille36/be047b523d95bdbf7e0e66e5c1d81298/colabsteam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Mobile Play + Colab Setup { display-mode: "form" }
from IPython.display import display, HTML
from google.colab import drive
import os
import subprocess
import threading
import time

# Keep Colab alive while playing
display(HTML('''
<b>Keep this audio playing to reduce Colab disconnects</b><br/>
<audio autoplay="" muted loop controls src="https://github.com/kmille36/Colab-Cloud-Gaming/raw/refs/heads/main/silence.m4a"></audio>
<script>
(function() {
  const audio = document.querySelector('audio');
  const keepAlive = () => {
    if (audio) {
      audio.volume = 0.01;
      audio.play().catch(() => {});
    }
    document.dispatchEvent(new Event('mousemove'));
    document.body && document.body.dispatchEvent(new Event('mousemove'));
    window.parent && window.parent.dispatchEvent(new Event('mousemove'));
  };
  setInterval(keepAlive, 15000);
  keepAlive();
})();
</script>
'''))

# Start a background heartbeat to keep the Colab session active
try:
    from google.colab import output
except Exception:
    output = None


def start_keepalive():
    def loop():
        while True:
            try:
                if output is not None:
                    output.eval_js("""
                    try {
                      document.dispatchEvent(new Event('mousemove'));
                      document.body && document.body.dispatchEvent(new Event('mousemove'));
                      window.parent && window.parent.dispatchEvent(new Event('mousemove'));
                    } catch (e) {}
                    """)
            except Exception:
                pass
            time.sleep(20)

    threading.Thread(target=loop, daemon=True).start()


start_keepalive()

# Mount Google Drive (optional)
Mount_Google_Drive = False #@param {type:"boolean"}
if Mount_Google_Drive:
    drive.mount('/content/drive')
    print("✅ Drive mounted.")
else:
    print("⛔ Drive mount skipped.")

# Show Colab VM region
region = subprocess.check_output(
    "curl -s ipinfo.io | grep region | cut -d\\\" -f4",
    shell=True,
    text=True
).strip()
print(f"Colab Region: {region or 'unknown'}")

# Download and run the main script in a detached session
subprocess.run(
    "wget -q https://github.com/kmille36/Colab-Cloud-Gaming/raw/refs/heads/main/ColabSteam -O /content/ColabSteam && chmod +x /content/ColabSteam",
    shell=True,
    check=True,
)

subprocess.Popen(
    ["/content/ColabSteam"],
    stdout=open('/tmp/colabsteam.log', 'a', encoding='utf-8'),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# Wait briefly for startup and print connection info
for _ in range(20):
    time.sleep(1)
    if os.path.exists('/tmp/colabsteam.log') and os.path.getsize('/tmp/colabsteam.log') > 0:
        break


def get_ip():
    candidates = [
        "hostname -I | awk '{print $1}'",
        "curl -s ifconfig.me || true",
        "curl -s https://api.ipify.org || true"
    ]
    for cmd in candidates:
        try:
            value = subprocess.check_output(cmd, shell=True, text=True).strip()
            if value and value != 'None':
                return value
        except Exception:
            pass
    return '127.0.0.1'

ip = get_ip()
print("\n🔗 Moonlight connection info")
print(f"  Host/IP: {ip}")
print("  Port: 47989")
print(f"  Web UI: https://{ip}:47990")
print("  Use the host above in Moonlight after the server starts.\n")